# Outreach Mailer (interactive)

Supervised batch sender: run cells top to bottom. Cell 5 is a **dry run** (prints what would be sent, sends nothing). Cell 6 actually sends.

Reads all credentials from a `.env` file next to this notebook (copy `.env.example` -> `.env` and fill in your own values — never commit the real `.env`).

**Rows only send inside their own Tue/Thu 7:30-9:30 AM recipient-local window** (from each row's `timezone` column) — never the sender's local time. A row outside that window right now is skipped and picked up automatically on a later run; nothing is ever sent early.

**Before raising the batch size past ~10:** check your inbox for bounces, not just this notebook's own success count. SMTP accepting a message only means the relay took it — a bad address bounces back separately, a few minutes later.

**Keep the per-send delay at 60s or more** on a shared institutional relay — the account behind it is often also your primary login, and a burst of sends in a few minutes is what gets a shared relay rate-limited or suspended.


In [ ]:
import smtplib, imaplib, ssl, csv, os, time, sys, email
from datetime import datetime, date
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from email.charset import Charset, QP
from email.utils import formataddr, formatdate, make_msgid
from collections import Counter
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# make the sibling `schedule` package importable (this notebook lives in src/send/)
NOTEBOOK_DIR = Path(os.path.abspath(''))
sys.path.insert(0, str(NOTEBOOK_DIR.parent))
from schedule.scheduling import is_due_now

UTF8_QP = Charset('utf-8')
UTF8_QP.body_encoding = QP
print(f'Python {sys.version.split()[0]} - imports OK')

In [ ]:
# ---- config: change BATCH for each new batch, everything else comes from .env ----
BATCH = 'batch_01'
CSV_PATH = f'{BATCH}.csv'
SENT_LOG = f'sent_log_{BATCH}.csv'

SENDER_NAME  = os.environ['SENDER_NAME']
SENDER_EMAIL = os.environ['SENDER_EMAIL']
SENDER_TOKEN = os.environ['SENDER_APP_TOKEN']
SMTP_SERVER  = os.environ['SMTP_SERVER']
SMTP_PORT    = int(os.environ.get('SMTP_PORT', 587))
IMAP_SERVER  = os.environ.get('IMAP_SERVER', '')
IMAP_PORT    = int(os.environ.get('IMAP_PORT', 993))
SAVE_TO_SENT = os.environ.get('SAVE_TO_SENT', 'false').lower() == 'true'
# fallback only — each row's own `resume_path` column (domain-specific resume) wins
# when present; RESUME_PATH is just what's used if a row somehow has none.
RESUME_PATH  = Path(os.environ.get('RESUME_PATH', './resume.pdf'))

DELAY_SECONDS = 60
LIMIT = None          # send only the first N this run, or None for the whole batch
RECONNECT_EVERY = 20

ctx = ssl.create_default_context()
print(f'batch={BATCH}  sender={SENDER_NAME} <{SENDER_EMAIL}>  delay={DELAY_SECONDS}s  save_to_sent={SAVE_TO_SENT}')

In [ ]:
# ---- load the batch CSV, skip already-sent rows and rows outside their Tue/Thu window ----
RESUME_OK = RESUME_PATH.exists() and open(RESUME_PATH, 'rb').read(5) == b'%PDF-'
print(f'fallback resume: {RESUME_PATH.name}  valid_pdf={RESUME_OK}  (per-row resume_path is used when present)')

LOGGED, already = [], set()
if os.path.exists(SENT_LOG):
    with open(SENT_LOG, newline='', encoding='utf-8') as f:
        LOGGED = list(csv.DictReader(f))
    already = {r['email'] for r in LOGGED if r.get('status') == 'sent'}
print(f'already sent in previous runs: {len(already)}')

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f'{CSV_PATH} not found next to this notebook.')

ALL, QUEUE, SKIPPED = [], [], []
with open(CSV_PATH, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        ALL.append(row)
        if row.get('send', '').strip().lower() != 'yes':
            SKIPPED.append((row['email'], 'send != yes')); continue
        if row['email'] in already:
            SKIPPED.append((row['email'], 'already sent')); continue
        if not (row.get('subject') and row.get('email_body') and row.get('resume_path')):
            SKIPPED.append((row['email'], 'missing content')); continue
        tz_name = (row.get('timezone') or '').strip()
        if not tz_name:
            SKIPPED.append((row['email'], 'missing timezone - cannot verify send window')); continue
        try:
            due = is_due_now(tz_name)
        except ValueError as e:
            SKIPPED.append((row['email'], f'invalid timezone: {e}')); continue
        if not due:
            SKIPPED.append((row['email'], f'outside Tue/Thu 7:30-9:30 AM local window ({tz_name})')); continue
        QUEUE.append(row)

if LIMIT:
    QUEUE = QUEUE[:LIMIT]
print(f'rows: {len(ALL)}  skipped: {len(SKIPPED)}  queued: {len(QUEUE)}')
if SKIPPED:
    for email, reason in SKIPPED[:15]:
        print(f'  skip {email}: {reason}')
print('next up:', [r['email'] for r in QUEUE[:5]])

In [ ]:
# ---- optional: find your IMAP Sent folder, so sends can be mirrored there ----
# Some networks restrict IMAP to on-campus/VPN access even when SMTP sending works
# fine from anywhere. If this cell fails or finds nothing, that's fine — sending
# still works via Cell 6, you just won't get an automatic Sent-folder copy.
SENT_FOLDER = None
if SAVE_TO_SENT and IMAP_SERVER:
    try:
        im = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT, ssl_context=ctx)
        im.login(SENDER_EMAIL, SENDER_TOKEN)
        ok, raw = im.list()
        names = [f.decode(errors='replace') for f in raw or []]
        for n in names:
            if '\\Sent' in n:
                SENT_FOLDER = n.split('"')[-2] if '"' in n else n.split()[-1]
                break
        print('Sent folder:', SENT_FOLDER or 'not found by \\Sent flag - check raw listing below')
        for n in names: print(' ', n)
        im.logout()
    except Exception as e:
        print(f'IMAP unavailable ({type(e).__name__}: {e}) - sending will still work, just without a Sent copy.')
else:
    print('SAVE_TO_SENT is off or IMAP_SERVER not set - skipping.')

In [ ]:
# ---- build functions + DRY RUN (prints what would be sent, sends nothing) ----
def build_html(body):
    import html as _h
    paras = [p.strip() for p in body.split('\n\n') if p.strip()]
    return ('<div style="font-family:sans-serif;font-size:14px;line-height:1.5">'
            + ''.join(f'<p>{_h.escape(p).replace(chr(10), "<br>")}</p>' for p in paras) + '</div>')

def resume_for(row):
    """The domain-specific resume for this row (Finance / FinTech / AI_ML / ...),
    falling back to the global RESUME_PATH only if the row has none."""
    raw = (row.get('resume_path') or '').strip()
    path = Path(raw) if raw else RESUME_PATH
    return path if path.exists() else None

def build_msg(row):
    attach = resume_for(row)
    if attach:
        msg, alt = MIMEMultipart('mixed'), MIMEMultipart('alternative')
    else:
        msg = MIMEMultipart('alternative'); alt = msg
    msg['From'] = formataddr((SENDER_NAME, SENDER_EMAIL))
    msg['To'] = row['email']
    msg['Subject'] = row['subject']
    msg['Reply-To'] = SENDER_EMAIL
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    body = row['email_body']
    if not body.endswith('\n'): body += '\n'
    alt.attach(MIMEText(body, 'plain', UTF8_QP))
    alt.attach(MIMEText(build_html(body), 'html', UTF8_QP))
    if attach:
        msg.attach(alt)
        part = MIMEApplication(attach.read_bytes(), _subtype='pdf')
        part.add_header('Content-Disposition', 'attachment', filename=attach.name)
        msg.attach(part)
    return msg

def append_to_sent(msg):
    if not (SAVE_TO_SENT and SENT_FOLDER): return False
    im = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT, ssl_context=ctx)
    try:
        im.login(SENDER_EMAIL, SENDER_TOKEN)
        im.append(SENT_FOLDER, '\\Seen', imaplib.Time2Internaldate(time.time()), bytes(msg))
        return True
    finally:
        try: im.logout()
        except Exception: pass

def log(row, status, err=''):
    exists = os.path.exists(SENT_LOG)
    fields = ['timestamp','date','batch','name','email','subject','status','error']
    with open(SENT_LOG, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if not exists: w.writeheader()
        w.writerow({'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'date': date.today().isoformat(), 'batch': BATCH,
                    'name': row.get('name',''), 'email': row['email'],
                    'subject': row['subject'], 'status': status, 'error': err})

print('='*70); print('DRY RUN - nothing sent'); print('='*70)
for i, row in enumerate(QUEUE[:3], 1):
    m = build_msg(row)
    print(f'\n[{i}] To: {m["To"]}  Subject: {m["Subject"]}  TZ: {row.get("timezone", "?")}')
    print(f'    Attach: {[p.get_filename() for p in m.walk() if p.get_filename()]}')
    print('-'*70); print(row['email_body'])
print(f'\n{len(QUEUE)} queued. Next cell sends them.')

In [ ]:
# ============================ SEND ============================
if not QUEUE:
    raise ValueError('Nothing queued - check the previous cells.')

missing_resume = [r['email'] for r in QUEUE if resume_for(r) is None]
if missing_resume:
    print(f'WARNING: no resolvable resume for {len(missing_resume)} queued row(s) (row resume_path '
          f'missing/not found, and no fallback RESUME_PATH): {missing_resume}')
    print('Sending WITHOUT a resume attached for those rows. Interrupt now if that is wrong.\n')
    time.sleep(5)

def smtp_connect():
    s = smtplib.SMTP(SMTP_SERVER, SMTP_PORT, timeout=30)
    s.ehlo(); s.starttls(context=ctx); s.ehlo()
    s.login(SENDER_EMAIL, SENDER_TOKEN)
    return s

print(f'Connecting as {SENDER_EMAIL}...')
s = smtp_connect()
print('Connected.\n' + '='*70)

ok = fail = skipped_late = saved_n = 0
try:
    for i, row in enumerate(QUEUE, 1):
        # re-check live: if review took long enough that this row fell out of its
        # Tue/Thu 7:30-9:30 window since the queue was built, do not send it late.
        tz_name = (row.get('timezone') or '').strip()
        if not tz_name or not is_due_now(tz_name):
            skipped_late += 1
            print(f'[{i}/{len(QUEUE)}] {row["email"]}: SKIPPED (no longer within its Tue/Thu 7:30-9:30 local window)')
            continue
        if i > 1 and (i - 1) % RECONNECT_EVERY == 0:
            try: s.quit()
            except Exception: pass
            s = smtp_connect()
        print(f'[{i}/{len(QUEUE)}] {row.get("name","")[:28]:<30} {row["email"]}')
        try:
            msg = build_msg(row)
            s.send_message(msg)
            ok += 1
            note = ''
            try:
                if append_to_sent(msg): saved_n += 1; note = ' + saved to Sent'
            except Exception as e:
                note = f' (Sent copy failed: {type(e).__name__})'
            print(f'         SENT{note}')
            log(row, 'sent')
        except smtplib.SMTPRecipientsRefused as e:
            fail += 1; print(f'         REFUSED: {e.recipients}'); log(row, 'refused', str(e.recipients))
        except Exception as e:
            fail += 1; print(f'         FAILED: {type(e).__name__}: {e}'); log(row, 'failed', str(e))
        if i < len(QUEUE): time.sleep(DELAY_SECONDS)
finally:
    try: s.quit()
    except Exception: pass

print('='*70)
print(f'sent: {ok}   failed: {fail}   skipped (fell out of window): {skipped_late}   copies in Sent: {saved_n}')
print(f'log: {SENT_LOG}')

In [ ]:
# ---- simple dashboard ----
if os.path.exists(SENT_LOG):
    with open(SENT_LOG, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    st = Counter(r['status'] for r in rows)
    print(f'{BATCH}: sent={st.get("sent",0)}  refused={st.get("refused",0)}  failed={st.get("failed",0)}')
    for r in sorted(rows, key=lambda x: x['timestamp'], reverse=True)[:15]:
        print(f'  {r["timestamp"]}  {r["email"]:<36} {r["status"]}')
else:
    print('No sends logged yet.')

## Next batch

Save it as `batch_02.csv` (same columns: `send, name, email, subject, email_body, resume_path, timezone`), set `BATCH = 'batch_02'` in the config cell, and re-run from the top. Each batch logs to its own `sent_log_<batch>.csv`, so batches never interfere with each other, and re-running a batch never double-sends.
